# 작은 행렬 곱셈

성능이 낮은 장난감 수준의 행렬 곱셈 구현입니다. `BLOCK_SIZE`가 행렬의 내적 차원보다 클 때만 동작합니다. `X` 행렬의 행 전체와 `Y` 행렬의 열 전체를 불러와 서로 곱합니다.

행렬 곱셈을 구현하는 더 나은 방법은 많습니다. 이 예제는 학습을 위한 것입니다.

먼저 행렬을 곱하는 Python 함수를 정의하겠습니다.

1. 올바른 크기와 디바이스로 출력 텐서를 초기화합니다.
2. Triton의 블록 크기는 2의 거듭제곱이어야 하므로, 행렬의 내적 차원보다 크거나 같은 다음 2의 거듭제곱을 선택합니다.
3. [문서](https://triton-lang.org/main/getting-started/tutorials/02-fused-softmax.html)에 따라 블록 크기에 맞춰 워프 수를 설정하면 성능을 높일 수 있습니다.
4. 행과 열을 순회하는 2차원 실행 그리드를 설정합니다. 유휴 스레드가 많기 때문에 비효율적입니다.
5. 데이터를 더 효율적으로 불러올 수 있도록 `Y` 행렬을 전치합니다. `contiguous` 호출은 메모리에서 `Y`의 배치도 바꾸므로 중요합니다.
6. 마지막으로 행렬 곱셈 커널을 호출합니다.

In [ ]:
import torch

import triton
import triton.language as tl

def matmul(X, Y):
    x_rows, x_cols = X.shape
    y_rows, y_cols = Y.shape
    output = torch.empty(x_rows, y_cols, device="cuda") # 출력 행렬
    
    # 블록 크기는 X의 열 수보다 크거나 같은 2의 거듭제곱으로 설정
    # X 행과 Y 열을 모두 불러올 크기로 선택
    BLOCK_SIZE = triton.next_power_of_2(x_cols)
    
    # 블록 크기가 클수록 연산을 빠르게 처리하도록 워프 수를 늘림
    num_warps = 4
    if BLOCK_SIZE >= 2048:
        num_warps = 8
    if BLOCK_SIZE >= 4096:
        num_warps = 16
    
    # 행과 열을 순회하는 2차원 그리드 생성
    grid = lambda meta: (x_rows, y_cols) # 행과 열을 순회하는 2차원 실행 그리드
    
    Y = Y.T.contiguous() # Y를 전치해 열 전체를 한 번에 불러옴. contiguous 호출은 메모리에서 텐서 배치도 변경
    matmul_kernel[grid](X, Y, output, x_rows, x_cols, y_rows, y_cols, BLOCK_SIZE=BLOCK_SIZE, num_warps=num_warps)
    
    return output

이제 커널을 작성하겠습니다. 실행 그리드에 따라 `X`의 각 행과 `Y`의 각 열 쌍마다 별도의 스레드 블록을 사용합니다.

1. 시작 포인터를 구합니다.
2. 행과 열의 모든 요소를 가리키는 포인터를 구합니다.
3. 범위를 벗어난 요소를 불러오지 않도록 마스크를 만듭니다.
4. 행과 열을 불러옵니다.
5. 서로 곱합니다.
6. 결과를 저장합니다.

In [ ]:
@triton.jit
def matmul_kernel(
    x_ptr,
    y_ptr,
    output_ptr,
    x_rows,
    x_cols,
    y_rows,
    y_cols,
    BLOCK_SIZE: tl.constexpr
):
    x_row_id = tl.program_id(axis=0) # 곱셈에 사용할 X의 행 위치
    y_col_id = tl.program_id(axis=1) # 곱셈에 사용할 Y의 열 위치
    
    x_row_start = x_row_id * x_cols # X에서 곱할 행의 첫 번째 인덱스
    y_col_start = y_col_id * y_rows # Y에서 곱할 열의 첫 번째 인덱스
    
    x_row_offset = x_row_start + tl.arange(0, BLOCK_SIZE) # X 행의 모든 요소 포인터
    x_mask = x_row_offset < x_row_start + x_cols # 마지막 열 이후의 X 데이터는 제외
    
    y_col_offset = y_col_start + tl.arange(0, BLOCK_SIZE) # Y 열의 모든 요소 포인터
    y_mask = y_col_offset < y_col_start + y_rows # 마지막 행 이후의 Y 데이터는 제외
    
    x_row = tl.load(x_ptr + x_row_offset, mask=x_mask, other=0.0) # 행 불러오기
    y_col = tl.load(y_ptr + y_col_offset, mask=y_mask, other=0.0) # 열 불러오기
    
    output = tl.sum(x_row * y_col, axis=0) # 요소별로 곱한 뒤 합산
     
    output_offset = (x_row_id * y_cols) + y_col_id # 결과 숫자 하나만 저장
    
    tl.store(output_ptr + output_offset, output)

앞서와 마찬가지로 행렬을 곱하고 결과를 확인합니다. 디버깅하기 쉽도록 `arange`를 사용해 순서가 있는 값을 만듭니다.

In [ ]:
torch.manual_seed(0)
x = torch.arange(12, device='cuda', dtype=torch.float32).reshape(4,3)
y = torch.arange(6, device='cuda', dtype=torch.float32).reshape(3,2)
output_torch = x @ y
output_triton = matmul(x, y)
print(
    f'PyTorch와 Triton 결과의 최댓값 차이: '
    f'{torch.max(torch.abs(output_torch - output_triton))}'
)
print(output_torch[:10])
print(output_triton[:10])

The maximum difference between torch and triton is 0.0
tensor([[10., 13.],
        [28., 40.],
        [46., 67.],
        [64., 94.]], device='cuda:0')
tensor([[10., 13.],
        [28., 40.],
        [46., 67.],
        [64., 94.]], device='cuda:0')
